# Phase 4: Exit Management with Trailing Stop (v68)

**Pine v68 Exit Flow:**

1. **Initial Stop** — Set at entry
2. **Break-Even (0.5R)** — Move stop to entry, EMA exit enabled
3. **Trailing Activation (2.0R)** — Start trailing at 1.2 ATR, EMA exit disabled
4. **Trailing Update** — Ratchet tighter each bar (never looser)
5. **Aggressive Mode** — Near EMA, tighten to 0.3 ATR

**Key insight:** Once trailing activates, it supersedes everything.

In [14]:
# Cell 1: Setup
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv(r'C:\Users\phemm\orb_lab\.env')

from data_collector import PolygonDataCollector

collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=5, bar_size=1)  # Recent data for today's trade
print(f"✓ Loaded {len(df)} bars")
print(f"Date range: {df.index[0]} to {df.index[-1]}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 5 days)...
  ✓ Loading from cache (0.2 hours old)
✓ Loaded 993 bars
Date range: 2025-12-24 09:30:00-05:00 to 2025-12-29 16:00:00-05:00


In [15]:
# Cell 2: Calculate indicators

def calc_atr(df, period=14):
    df = df.copy()
    tr1 = df['high'] - df['low']
    tr2 = abs(df['high'] - df['close'].shift(1))
    tr3 = abs(df['low'] - df['close'].shift(1))
    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['atr'] = df['tr'].ewm(alpha=1/period, adjust=False).mean()
    return df

def calc_ema(df, period=9, col_name='ema9'):
    df = df.copy()
    df[col_name] = df['close'].ewm(span=period, adjust=False).mean()
    return df

df = calc_atr(df)
df = calc_ema(df, period=9, col_name='ema9')
print("✓ ATR and EMA calculated")

✓ ATR and EMA calculated


In [16]:
# Cell 3: Complete Exit Management Class

class ExitManager:
    """
    Pine v68 Exit Management:
    - Break-even at 0.5R
    - EMA exit after BE (disabled when trailing active)
    - Trailing stop at 2.0R profit target
    - Aggressive trailing near EMA
    """
    
    def __init__(self, 
                 break_even_rr=0.5,
                 profit_target_rr=2.0,
                 trailing_stop_atr=1.2,
                 ema_confirmation_bars=1,
                 aggressive_trailing=True,
                 ema_tighten_zone=0.3,
                 tightened_trail_atr=0.3):
        
        self.break_even_rr = break_even_rr
        self.profit_target_rr = profit_target_rr
        self.trailing_stop_atr = trailing_stop_atr
        self.ema_confirmation_bars = ema_confirmation_bars
        self.aggressive_trailing = aggressive_trailing
        self.ema_tighten_zone = ema_tighten_zone
        self.tightened_trail_atr = tightened_trail_atr
        
        # State
        self.reset()
    
    def reset(self):
        """Reset for new trade"""
        self.be_triggered = False
        self.be_bar = None
        self.trailing_activated = False
        self.trailing_bar = None
        self.trailing_stop_level = None
        self.ema_cross_count = 0
        self.current_stop = None
    
    def initialize_trade(self, entry_price, initial_stop, is_long):
        """Set up for a new trade"""
        self.reset()
        self.entry_price = entry_price
        self.initial_stop = initial_stop
        self.is_long = is_long
        self.risk = abs(entry_price - initial_stop)
        self.current_stop = initial_stop
        
        # Calculate targets
        if is_long:
            self.be_target = entry_price + (self.risk * self.break_even_rr)
            self.profit_target = entry_price + (self.risk * self.profit_target_rr)
        else:
            self.be_target = entry_price - (self.risk * self.break_even_rr)
            self.profit_target = entry_price - (self.risk * self.profit_target_rr)
    
    def update(self, bar_idx, row, atr):
        """
        Process one bar. Returns exit signal if triggered.
        
        Returns: dict with keys:
            - 'exit': bool
            - 'exit_price': float or None
            - 'exit_reason': str or None
            - 'current_stop': float
            - 'be_triggered': bool
            - 'trailing_activated': bool
            - 'trailing_level': float or None
        """
        close = row['close']
        high = row['high']
        low = row['low']
        ema9 = row['ema9']
        
        result = {
            'exit': False,
            'exit_price': None,
            'exit_reason': None,
            'current_stop': self.current_stop,
            'be_triggered': self.be_triggered,
            'trailing_activated': self.trailing_activated,
            'trailing_level': self.trailing_stop_level
        }
        
        # 1. CHECK STOP HIT FIRST
        if self.is_long:
            if low <= self.current_stop:
                result['exit'] = True
                result['exit_price'] = self.current_stop
                if self.trailing_activated:
                    result['exit_reason'] = 'Trail Stop'
                elif self.be_triggered:
                    result['exit_reason'] = 'Break Even'
                else:
                    result['exit_reason'] = 'Stop Loss'
                return result
        else:
            if high >= self.current_stop:
                result['exit'] = True
                result['exit_price'] = self.current_stop
                if self.trailing_activated:
                    result['exit_reason'] = 'Trail Stop'
                elif self.be_triggered:
                    result['exit_reason'] = 'Break Even'
                else:
                    result['exit_reason'] = 'Stop Loss'
                return result
        
        # 2. CHECK BREAK-EVEN TRIGGER
        if not self.be_triggered:
            if self.is_long and close >= self.be_target:
                self.be_triggered = True
                self.be_bar = bar_idx
                self.current_stop = self.entry_price
            elif not self.is_long and close <= self.be_target:
                self.be_triggered = True
                self.be_bar = bar_idx
                self.current_stop = self.entry_price
        
        # 3. CHECK TRAILING ACTIVATION (requires BE first)
        if self.be_triggered and not self.trailing_activated:
            if self.is_long and close >= self.profit_target:
                self.trailing_activated = True
                self.trailing_bar = bar_idx
                self.trailing_stop_level = close - (atr * self.trailing_stop_atr)
                self.current_stop = max(self.trailing_stop_level, self.entry_price)
            elif not self.is_long and close <= self.profit_target:
                self.trailing_activated = True
                self.trailing_bar = bar_idx
                self.trailing_stop_level = close + (atr * self.trailing_stop_atr)
                self.current_stop = min(self.trailing_stop_level, self.entry_price)
        
        # 4. UPDATE TRAILING STOP (ratchet only)
        if self.trailing_activated:
            # Check for aggressive trailing near EMA
            trail_distance = self.trailing_stop_atr
            if self.aggressive_trailing:
                dist_to_ema = abs(close - ema9)
                if dist_to_ema <= (atr * self.ema_tighten_zone):
                    trail_distance = self.tightened_trail_atr
            
            if self.is_long:
                new_trail = close - (atr * trail_distance)
                if new_trail > self.trailing_stop_level:
                    self.trailing_stop_level = new_trail
                self.current_stop = max(self.trailing_stop_level, self.entry_price)
            else:
                new_trail = close + (atr * trail_distance)
                if new_trail < self.trailing_stop_level:
                    self.trailing_stop_level = new_trail
                self.current_stop = min(self.trailing_stop_level, self.entry_price)
        
        # 5. EMA EXIT (only if BE triggered but trailing NOT active)
        if self.be_triggered and not self.trailing_activated:
            in_profit = (self.is_long and close > self.entry_price) or \
                        (not self.is_long and close < self.entry_price)
            
            if in_profit:
                ema_crossed = (self.is_long and close < ema9) or \
                              (not self.is_long and close > ema9)
                
                if ema_crossed:
                    self.ema_cross_count += 1
                    if self.ema_cross_count >= self.ema_confirmation_bars:
                        result['exit'] = True
                        result['exit_price'] = close
                        result['exit_reason'] = 'EMA Exit'
                        return result
                else:
                    self.ema_cross_count = 0
            else:
                self.ema_cross_count = 0
        
        # Update result
        result['current_stop'] = self.current_stop
        result['be_triggered'] = self.be_triggered
        result['trailing_activated'] = self.trailing_activated
        result['trailing_level'] = self.trailing_stop_level
        
        return result

print("✓ ExitManager class defined")

✓ ExitManager class defined


In [17]:
# Cell 4: Trade Simulator with full exit management

def simulate_trade(df, entry_idx, entry_price, initial_stop, is_long,
                   break_even_rr=0.5, profit_target_rr=2.0, trailing_stop_atr=1.2,
                   ema_confirmation=1, aggressive_trailing=True,
                   ema_tighten_zone=0.3, tightened_trail_atr=0.3):
    """
    Simulate a complete trade with v68 exit management.
    Returns detailed trade log and result.
    """
    
    exit_mgr = ExitManager(
        break_even_rr=break_even_rr,
        profit_target_rr=profit_target_rr,
        trailing_stop_atr=trailing_stop_atr,
        ema_confirmation_bars=ema_confirmation,
        aggressive_trailing=aggressive_trailing,
        ema_tighten_zone=ema_tighten_zone,
        tightened_trail_atr=tightened_trail_atr
    )
    
    exit_mgr.initialize_trade(entry_price, initial_stop, is_long)
    
    trade_log = []
    bar_data = []  # For detailed tracking
    
    trade_log.append({
        'time': df.index[entry_idx],
        'event': f'ENTRY {"LONG" if is_long else "SHORT"} @ {entry_price:.2f}',
        'stop': initial_stop,
        'trailing': None
    })
    
    for i in range(entry_idx + 1, len(df)):
        row = df.iloc[i]
        atr = row['atr']
        bar_time = df.index[i]
        
        result = exit_mgr.update(i, row, atr)
        
        # Log significant events
        if result['be_triggered'] and exit_mgr.be_bar == i:
            trade_log.append({
                'time': bar_time,
                'event': f'BE TRIGGERED - stop moved to {exit_mgr.entry_price:.2f}',
                'stop': result['current_stop'],
                'trailing': None
            })
        
        if result['trailing_activated'] and exit_mgr.trailing_bar == i:
            trade_log.append({
                'time': bar_time,
                'event': f'TRAIL ACTIVATED @ {result["trailing_level"]:.2f}',
                'stop': result['current_stop'],
                'trailing': result['trailing_level']
            })
        
        # Track bar data
        bar_data.append({
            'time': bar_time,
            'close': row['close'],
            'high': row['high'],
            'low': row['low'],
            'ema9': row['ema9'],
            'atr': atr,
            'current_stop': result['current_stop'],
            'trailing_level': result['trailing_level'],
            'be_triggered': result['be_triggered'],
            'trailing_activated': result['trailing_activated']
        })
        
        if result['exit']:
            trade_log.append({
                'time': bar_time,
                'event': f'{result["exit_reason"]} @ {result["exit_price"]:.2f}',
                'stop': result['current_stop'],
                'trailing': result['trailing_level']
            })
            break
    
    # Calculate P&L
    exit_price = result.get('exit_price')
    if exit_price:
        if is_long:
            pnl = exit_price - entry_price
        else:
            pnl = entry_price - exit_price
        risk = abs(entry_price - initial_stop)
        pnl_r = pnl / risk if risk > 0 else 0
    else:
        pnl = None
        pnl_r = None
    
    return {
        'entry_price': entry_price,
        'initial_stop': initial_stop,
        'risk': abs(entry_price - initial_stop),
        'be_target': exit_mgr.be_target,
        'profit_target': exit_mgr.profit_target,
        'be_triggered': exit_mgr.be_triggered,
        'trailing_activated': exit_mgr.trailing_activated,
        'exit_price': exit_price,
        'exit_reason': result.get('exit_reason'),
        'pnl': pnl,
        'pnl_r': pnl_r,
        'trade_log': trade_log,
        'bar_data': pd.DataFrame(bar_data)
    }

print("✓ Trade simulator defined")

✓ Trade simulator defined


In [18]:
# Cell 5: Find today's AMD trade and simulate

# Get today's date
today = df.index[-1].date()
print(f"Looking for trades on: {today}")

# Filter to today
today_df = df[df.index.date == today].copy()
print(f"Bars today: {len(today_df)}")

# Show the first hour
print("\nFirst hour data:")
first_hour = today_df.between_time('09:30', '10:30')
print(first_hour[['open', 'high', 'low', 'close', 'atr', 'ema9']].head(15))

Looking for trades on: 2025-12-29
Bars today: 391

First hour data:
                              open      high       low     close       atr  \
timestamp                                                                    
2025-12-29 09:30:00-05:00  211.575  211.6600  209.6277  209.9850  0.514226   
2025-12-29 09:31:00-05:00  209.890  210.0899  209.2400  209.7400  0.538203   
2025-12-29 09:32:00-05:00  209.840  210.9200  209.7600  210.5800  0.584045   
2025-12-29 09:33:00-05:00  210.620  210.8856  210.2500  210.5500  0.587728   
2025-12-29 09:34:00-05:00  210.510  211.6000  210.1200  211.5230  0.651462   
2025-12-29 09:35:00-05:00  211.510  211.8173  211.0100  211.2290  0.662593   
2025-12-29 09:36:00-05:00  211.400  213.3999  211.4000  213.2010  0.770329   
2025-12-29 09:37:00-05:00  213.300  214.2613  213.2100  213.9668  0.791041   
2025-12-29 09:38:00-05:00  213.990  214.5200  213.8600  214.2100  0.781681   
2025-12-29 09:39:00-05:00  214.270  214.4500  213.7900  214.3550  0.772990

In [19]:
# Cell 6: Manual trade setup (adjust based on actual entry from TradingView)

# You'll need to input the actual entry details from your AMD trade
# Look at TradingView for the exact values

# Example setup - ADJUST THESE TO MATCH YOUR ACTUAL TRADE:
ENTRY_TIME = '10:19'  # Approximate entry time
ENTRY_PRICE = 212.44  # Your actual entry price
INITIAL_STOP = 212.31  # Your actual initial stop
IS_LONG = True  # True for long, False for short

# Find the entry bar
entry_bar = today_df.between_time(ENTRY_TIME, ENTRY_TIME)
if len(entry_bar) > 0:
    entry_idx = df.index.get_loc(entry_bar.index[0])
    print(f"Entry bar found at index {entry_idx}")
    print(f"Entry bar data: {df.iloc[entry_idx][['open', 'high', 'low', 'close']].to_dict()}")
else:
    # Fallback - use first bar after 9:35
    post_orb = today_df.between_time('09:36', '10:30')
    if len(post_orb) > 0:
        entry_idx = df.index.get_loc(post_orb.index[0])
        print(f"Using first post-ORB bar at index {entry_idx}")
        print(f"Adjust ENTRY_TIME, ENTRY_PRICE, INITIAL_STOP to match your actual trade")

Entry bar found at index 651
Entry bar data: {'open': 212.453833, 'high': 212.5, 'low': 212.2847, 'close': 212.445}


In [20]:
# Cell 7: Run the simulation

result = simulate_trade(
    df, 
    entry_idx=entry_idx,
    entry_price=ENTRY_PRICE,
    initial_stop=INITIAL_STOP,
    is_long=IS_LONG,
    break_even_rr=0.5,
    profit_target_rr=2.0,
    trailing_stop_atr=1.2,
    ema_confirmation=1,
    aggressive_trailing=True,
    ema_tighten_zone=0.3,
    tightened_trail_atr=0.3
)

print("=" * 60)
print("                    TRADE SIMULATION RESULT")
print("=" * 60)
print(f"\nEntry: ${result['entry_price']:.2f}")
print(f"Initial Stop: ${result['initial_stop']:.2f}")
print(f"Risk: ${result['risk']:.2f}")
print(f"\nBE Target (0.5R): ${result['be_target']:.2f}")
print(f"Profit Target (2.0R): ${result['profit_target']:.2f}")
print(f"\nBE Triggered: {result['be_triggered']}")
print(f"Trailing Activated: {result['trailing_activated']}")
print(f"\nExit Price: ${result['exit_price']:.2f}" if result['exit_price'] else "Trade still open")
print(f"Exit Reason: {result['exit_reason']}")
print(f"\nP&L: ${result['pnl']:.2f}" if result['pnl'] else "N/A")
print(f"P&L (R): {result['pnl_r']:.2f}R" if result['pnl_r'] else "N/A")

print("\n" + "=" * 60)
print("                         TRADE LOG")
print("=" * 60)
for log in result['trade_log']:
    time_str = log['time'].strftime('%H:%M') if hasattr(log['time'], 'strftime') else str(log['time'])
    print(f"{time_str}: {log['event']}")

                    TRADE SIMULATION RESULT

Entry: $212.44
Initial Stop: $212.31
Risk: $0.13

BE Target (0.5R): $212.50
Profit Target (2.0R): $212.70

BE Triggered: True
Trailing Activated: True

Exit Price: $213.89
Exit Reason: Trail Stop

P&L: $1.45
P&L (R): 11.15R

                         TRADE LOG
10:19: ENTRY LONG @ 212.44
10:20: BE TRIGGERED - stop moved to 212.44
10:21: TRAIL ACTIVATED @ 212.28
10:38: Trail Stop @ 213.89


In [21]:
# Cell 8: Show trailing stop progression

if result['trailing_activated']:
    bar_df = result['bar_data']
    trailing_bars = bar_df[bar_df['trailing_activated']].copy()
    
    print("\n" + "=" * 60)
    print("              TRAILING STOP PROGRESSION")
    print("=" * 60)
    print(f"\n{'Time':>8} | {'Close':>8} | {'Trail':>8} | {'Stop':>8} | {'EMA9':>8} | Notes")
    print("-" * 70)
    
    prev_trail = None
    for idx, row in trailing_bars.iterrows():
        time_str = row['time'].strftime('%H:%M')
        
        notes = []
        if prev_trail and row['trailing_level'] > prev_trail:
            notes.append('RATCHET ↑')
        
        # Check if in aggressive zone
        dist_to_ema = abs(row['close'] - row['ema9'])
        if dist_to_ema <= (row['atr'] * 0.3):
            notes.append('TIGHT')
        
        print(f"{time_str:>8} | {row['close']:>8.2f} | {row['trailing_level']:>8.2f} | {row['current_stop']:>8.2f} | {row['ema9']:>8.2f} | {' '.join(notes)}")
        prev_trail = row['trailing_level']
else:
    print("\nTrailing stop was not activated in this trade.")


              TRAILING STOP PROGRESSION

    Time |    Close |    Trail |     Stop |     EMA9 | Notes
----------------------------------------------------------------------
   10:21 |   212.77 |   212.28 |   212.44 |   212.30 | 
   10:22 |   212.93 |   212.45 |   212.45 |   212.42 | RATCHET ↑
   10:23 |   213.01 |   212.55 |   212.55 |   212.54 | RATCHET ↑
   10:24 |   213.26 |   212.80 |   212.80 |   212.68 | RATCHET ↑
   10:25 |   213.43 |   212.97 |   212.97 |   212.83 | RATCHET ↑
   10:26 |   213.35 |   212.97 |   212.97 |   212.94 | 
   10:27 |   213.51 |   213.08 |   213.08 |   213.05 | RATCHET ↑
   10:28 |   213.49 |   213.08 |   213.08 |   213.14 | 
   10:29 |   213.62 |   213.20 |   213.20 |   213.24 | RATCHET ↑
   10:30 |   213.89 |   213.46 |   213.46 |   213.37 | RATCHET ↑
   10:31 |   213.62 |   213.46 |   213.46 |   213.42 | 
   10:32 |   213.87 |   213.46 |   213.46 |   213.51 | 
   10:33 |   213.82 |   213.46 |   213.46 |   213.57 | 
   10:34 |   213.96 |   213.56 |   

In [ ]:
# Cell 9: Validation checklist

print("""
═══════════════════════════════════════════════════════════════
                 PHASE 4 VALIDATION CHECKLIST
═══════════════════════════════════════════════════════════════

Compare Python simulation with TradingView v68 replay:

□ BE trigger time matches (look for 'BE' label)
□ Trailing activation time matches (look for 'TRAIL' label)  
□ Trailing stop level matches blue line on chart
□ Exit reason matches (Trail Stop / EMA Exit / Break Even)
□ Exit price matches
□ Final P&L in R matches

Exit Priority (v68):
─────────────────────
1. 0 to 0.5R    → Initial stop protects
2. 0.5R to 2.0R → BE stop, EMA exit CAN trigger
3. 2.0R+        → Trailing takes over, EMA exit DISABLED

═══════════════════════════════════════════════════════════════
""")

## Next Steps

Once Phase 4 validated:
- Update `orb_backtester.py` with `ExitManager` class
- Phase 5: Adaptive vol regime adjustments
- Then: Full backtest runs with Optuna optimization